<a href="https://colab.research.google.com/github/Manikanta-23-ai/LANGCHAIN-LEARNING/blob/main/PERSISTENT_DATABASE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
%pip install -q -U langchain-chroma langchain-huggingface sentence-transformers

In [6]:
from langchain_chroma import Chroma

print("ChromaDB imported successfully!")


ChromaDB imported successfully!


In [8]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/langchain_project")
PDF_DIR = PROJECT_DIR / "pdfs"
CHUNKS_DIR = PROJECT_DIR / "chunks"
DB_DIR = PROJECT_DIR / "chroma_db"

for folder in [PDF_DIR, CHUNKS_DIR, DB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Project folder: /content/drive/MyDrive/langchain_project


In [9]:
from google.colab import files
import shutil

uploaded = files.upload()
filename = next(iter(uploaded))

pdf_path = PDF_DIR / filename
shutil.move(filename, str(pdf_path))

print("PDF saved:", pdf_path)

Saving SQL_Notes_Premium.pdf to SQL_Notes_Premium (1).pdf
PDF saved: /content/drive/MyDrive/langchain_project/pdfs/SQL_Notes_Premium (1).pdf


In [10]:
%pip install -q -U langchain-community langchain-text-splitters pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [12]:
%pip install --upgrade pip

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(str(pdf_path)).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))

/tmp/ipykernel_768/3259390314.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages: 188
Chunks: 409


In [14]:
import json

chunks_data = [
    {
        "text": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in chunks
]

with open(CHUNKS_DIR / "pdf_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print("Chunks saved to Google Drive!")

Chunks saved to Google Drive!


In [15]:
%pip install -q -U langchain-chroma langchain-huggingface sentence-transformers

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="my_pdf",
    persist_directory=str(DB_DIR)
)

print("ChromaDB saved to Google Drive!")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB saved to Google Drive!


In [17]:
retriever = db.as_retriever(search_kwargs={"k": 3})

question = input("Ask a question about your PDF: ")
docs = retriever.invoke(question)

for doc in docs:
    print("\n", doc.page_content)
    print("Page:", doc.metadata.get("page", 0) + 1)

Ask a question about your PDF: what is sql'

 SQL is a standard language for accessing and manipulating databases.
SQL stands for Structured Query Language
SQL lets you access and manipulate databases
SQL is an ANSI (American National Standards Institute) standard
SQL can execute queries against a database
SQL can retrieve data from a database
SQL can insert records in a database
SQL can update records in a database
SQL can delete records from a database
SQL can create new databases
SQL can create new tables in a database
Page: 1

 Note: Most of the SQL database programs also have their own proprietary
extensions in addition to the SQL standard!
THE WORLD'S LARGEST WEB DEVELOPER SITE
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
1
Page: 1

 SQL statement, via web page input.
Injected SQL commands can alter SQL statement and compromise the security of a web
application.
THE WORLD'S LARGEST WEB DEVELOPER SITE
Created By www.ebooktutorials.in
Content Downloade